In [ ]:
# Importa o módulo os para acessar variáveis de ambiente.
import os

# Carrega as variáveis definidas no arquivo .env.
from dotenv import load_dotenv

# Biblioteca oficial para acessar a API do Google Gemini.
from google import genai

# Tipos de configuração usados para configurar a resposta do modelo.
from google.genai import types

# Carrega as variáveis do arquivo .env para o ambiente do Python.
load_dotenv()

# Recupera a chave da API do Gemini a partir da variável GEMINI_KEY.
# A chave real fica no arquivo .env e não deve ser publicada no GitHub.
GEMINI_KEY = os.getenv("GEMINI_KEY")

# Verifica se a chave foi configurada antes de criar o cliente.
if not GEMINI_KEY:
    raise ValueError(
        "A variável GEMINI_KEY não foi encontrada. "
        "Crie um arquivo .env e informe sua chave da API do Gemini."
    )

# Cria o cliente que será usado para enviar mensagens ao Gemini.
client = genai.Client(api_key=GEMINI_KEY)


In [ ]:
# Função responsável por enviar o histórico da conversa ao Gemini.
# O histórico é recebido em 'messages' para que o modelo mantenha o contexto.
def get_completion_from_messages(
    messages,
    model="gemini-3.6-flash",
    temperature=0
):
    response = client.models.generate_content(
        model=model,
        contents=messages,
        config=types.GenerateContentConfig(
            temperature=temperature
        )
    )
    return response.text

In [ ]:
# Função executada quando o usuário clica no botão Enviar.
# Ela captura a pergunta, atualiza o histórico, consulta o Gemini
# e mostra a resposta na interface.
import panel as pn
pn.extension()

panels = []

context = [
    {
        'role': 'user',
        'parts': [{'text': """
Você é o EducaReserve, um assistente virtual especializado
na reserva de equipamentos eletrônicos da Escola ABC.

PERSONALIDADE:
Você é educado, objetivo e profissional.
Responda de forma curta e clara.

OBJETIVO:
Seu objetivo é responder perguntas dos professores
sobre a disponibilidade e as regras de utilização
dos equipamentos eletrônicos da escola.

EQUIPAMENTOS DISPONÍVEIS:
- 30 tablets
- 20 notebooks
- 5 projetores

HORÁRIO DE FUNCIONAMENTO:
- Segunda a sexta: 07:00 às 18:00
- Sábado e domingo: não funciona.

TURMAS DISPONÍVEIS:
- 1º ano A
- 1º ano B
- 2º ano A
- 2º ano B
- 3º ano A
- 3º ano B

RESERVAS REGISTRADAS:

20/09/2026:
- 10 notebooks reservados das 08:00 às 10:00
  pelo professor Carlos para a turma 2º ano A.
- 15 tablets reservados das 10:00 às 12:00
  pela professora Ana para a turma 1º ano B.

21/09/2026:
- 20 notebooks reservados das 14:00 às 16:00
  pelo professor João para a turma 3º ano A.

REGRAS:
- Nunca informe uma quantidade diferente das informações
  fornecidas neste contexto.
- Não invente equipamentos, reservas, horários ou regras.
- Se a informação solicitada não estiver no contexto,
  diga que não possui essa informação.
- Para verificar disponibilidade, considere a quantidade
  total do equipamento e as reservas existentes no mesmo
  período.
- Uma reserva não pode ultrapassar a quantidade disponível.
- A turma precisa existir na escola.
- Reservas somente dentro do horário de funcionamento.
- Não são permitidas reservas aos finais de semana.

LIMITE DA CONVERSA:
Responda três perguntas do professor.
Após a terceira resposta, apresente um breve resumo
das três perguntas e respostas e encerre a conversa.

Não responda perguntas que não estejam relacionadas
ao contexto da Escola ABC.
"""}]
    },
    {
        'role': 'model',
        'parts': [{'text': "Bem vindo professor, sou o EducaReserva, vamos reservar equipamentos?"}]
    }
]

panels.append(
    pn.Row('Assistant:', pn.pane.Markdown("Bem vindo professor, sou o EducaReserva, vamos reservar equipamentos?", width=600, styles={'background-color': '#F6F6F6'}))
)

def collect_messages(_):
    prompt = inp.value
    inp.value = ''
    if not prompt.strip():
        return pn.Column(*panels)

    context.append({'role': 'user', 'parts': [{'text': prompt}]})
    response = get_completion_from_messages(context)
    context.append({'role': 'model', 'parts': [{'text': response}]})

    panels.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)

inp = pn.widgets.TextInput(placeholder='Digite sua mensagem aqui…')
button_conversation = pn.widgets.Button(name="Enviar")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=400),
)

dashboard


/tmp/ipykernel_1646/3040089174.py:2: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Digite sua mensagem a...)
    [1] Row
        [0] Button(label='Enviar', name='Enviar')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=400, loading_indicator=True)

**Perguntas para o exemplo:**

Quantos notebooks estão disponíveis no dia 20/09 das 08h às 10h?

Posso reservar 15 notebooks nesse horário para uma turma?

E das 10h às 12h?